# Notebook 05 — Validación Independiente (AREA2 + Collect Earth Online), F1-Score y Prueba de Hipótesis (Objetivo Específico 3)

- **Instancia SEPAL recomendada:** `t2` (2 CPU, 4 GB RAM, 0.04 USD/h).
- **Responde al Objetivo Específico 3 y contrasta la Hipótesis ($H_0$ vs. $H_1$):**
  *"Determinar el porcentaje de precisión obtenido en el modelo combinado de sensores multiespectral y SAR."*
- **Flujo de trabajo integrado con AREA2 y Collect Earth Online (CEO):**
  1. En **AREA2 (GEE)** generas el diseño de muestreo aleatorio estratificado a partir del Asset exportado en el Notebook 04 (`mapa_ms_sar_pca_<region>_v1_sepal` o `mapa_estratos_area2_<region>_v1_sepal`) y exportas el CSV de puntos en `EPSG:4326`.
  2. Subes ese único CSV a **Collect Earth Online (CEO)** e interpretas cada punto **una sola vez** con imágenes de alta resolución **Planet + Sentinel-2**.
  3. Descargas el CSV validado de CEO, lo colocas en `outputs/muestras/` y corres este notebook:
     - Extrae automáticamente la predicción de **todos los modelos (MS, SAR, MS+SAR PCA, MS+SAR Completo)** en los puntos de CEO.
     - Aplica los estimadores insesgados **Olofsson et al. (2014) / Stehman (2014)** para calcular:
       - Matrices de confusión en proporciones de área ($\hat{p}_{ij}$)
       - $\widehat{OA} \pm \text{IC 95\%}$, $\widehat{PU} \pm \text{IC 95\%}$, $\widehat{PP} \pm \text{IC 95\%}$
       - **F1-Score ponderado por área $\pm$ IC 95%** (bootstrap estratificado)
       - **Índice Kappa ($\hat{\kappa}$) $\pm$ IC 95%**
       - **Superficie Ajustada (Insesgada) de Manglar (ha) $\pm$ IC 95%**
     - Ejecuta **Prueba de McNemar**, **Prueba Z ponderada por diseño** y **Prueba Z de Kappa** para aceptar o rechazar $H_0$ ($\alpha = 0.05$).
     - Exporta todas las tablas (Excel + CSV) y figuras a 300 DPI listas para los Capítulos 2 y 3 de tu tesis.

In [ ]:
# 1. Configuracion y parametros del archivo exportado de Collect Earth Online (CEO)
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import Point

from src.config_utils import ensure_output_dirs, get_class_maps, init_gee, load_config
from src.metrics import (
    kappa_z_test,
    mcnemar_test,
    olofsson_accuracy_area,
    stehman_generalized_accuracy,
    stratified_paired_z_test,
)
from src.plotting import plot_confusion_matrix, plot_model_comparison, set_thesis_style

# >>> AJUSTAR ESTOS PARAMETROS CON TU ARCHIVO REAL DE COLLECT EARTH ONLINE <<<
REGION = "PACIFICO"
CEO_CSV_PATH = None  # Ejemplo: "../outputs/muestras/ceo_validacion_pacifico.csv"

# Nombres de columnas en el CSV exportado desde Collect Earth Online / AREA2
COL_LON = "lon"          # En CEO suele ser 'lon' o 'center_lon'
COL_LAT = "lat"          # En CEO suele ser 'lat' o 'center_lat'
COL_STRATUM = "stratum"  # Columna del estrato asignado por AREA2 (1..5 o 1..6)
COL_REF = "ref_class"    # Columna con tu etiqueta validada en CEO

# Tipo de mapa usado para estratificar en AREA2:
#   - "MS_SAR_PCA"        -> si usaste el mapa combinado de 5 clases en AREA2
#   - "ACUERDO_DESACUERDO" -> si usaste el mapa de 6 estratos de Acuerdo/Desacuerdo en AREA2
TIPO_ESTRATO_AREA2 = "MS_SAR_PCA"

# Traduccion de etiquetas de texto de CEO a codigos 1..5 (si ya son enteros 1..5 se mantienen igual)
MAPEO_ETIQUETAS_CEO = {
    "Mangle": 1, "mangle": 1, "1": 1, 1: 1,
    "Bosque latifoliado": 2, "Latifoliado": 2, "Bosque": 2, "2": 2, 2: 2,
    "Agua": 3, "agua": 3, "3": 3, 3: 3,
    "Infraestructura urbana": 4, "Urbano": 4, "urbano": 4, "4": 4, 4: 4,
    "Pastos/Ganadería": 5, "Pastos/Ganaderia": 5, "Pastos": 5, "5": 5, 5: 5,
}

cfg = load_config()
dirs = ensure_output_dirs(cfg)
init_gee(cfg)
set_thesis_style()

reg_l = REGION.lower()
ver = cfg["proyecto"]["version_pipeline"].lower()
CLASES_NOMBRES, CLASES_CORTOS, CLASES_COLORES = get_class_maps(cfg)
CLASS_IDS = sorted(CLASES_NOMBRES.keys())
val_cfg = cfg["validacion_olofsson"]

# Cargar areas exactas de estratos guardadas en el Notebook 04
areas_json = json.loads((dirs["tablas"] / f"areas_estratos_{reg_l}.json").read_text(encoding="utf-8"))
if TIPO_ESTRATO_AREA2 == "ACUERDO_DESACUERDO":
    STRATUM_AREAS_HA = {int(k): float(v) for k, v in areas_json["areas_estratos_acuerdo_desacuerdo_ha"].items()}
else:
    STRATUM_AREAS_HA = {int(k): float(v) for k, v in areas_json["areas_modelos_ha"][TIPO_ESTRATO_AREA2].items()}

print("Areas de estratos cargadas (ha):", {k: round(v, 2) for k, v in sorted(STRATUM_AREAS_HA.items())})

In [ ]:
# 2. Cargar CSV de Collect Earth Online y extraer predicciones de los 4 modelos desde GEE
assert CEO_CSV_PATH is not None and Path(CEO_CSV_PATH).exists(), (
    "Indica la ruta de tu CSV exportado de Collect Earth Online en CEO_CSV_PATH (celda 1)."
)

df_ceo = pd.read_csv(CEO_CSV_PATH).copy()
df_ceo["stratum_id"] = df_ceo[COL_STRATUM].astype(int)
df_ceo["ref_id"] = df_ceo[COL_REF].map(MAPEO_ETIQUETAS_CEO)
n_sin_mapear = int(df_ceo["ref_id"].isna().sum())
if n_sin_mapear > 0:
    print("Etiquetas sin mapear encontradas:", df_ceo.loc[df_ceo["ref_id"].isna(), COL_REF].unique())
    raise ValueError("Agrega las etiquetas faltantes a MAPEO_ETIQUETAS_CEO en la celda 1.")
df_ceo["ref_id"] = df_ceo["ref_id"].astype(int)
df_ceo["point_uid"] = np.arange(1, len(df_ceo) + 1)

# Construir imagen multibanda con los 4 mapas clasificados exportados en el Notebook 04
root_asset = cfg["gee"]["asset_root"].rstrip("/") + "/"
modelos_evaluar = ["MS", "SAR", "MS_SAR_PCA", "MS_SAR_FULL"]
img_modelos = ee.Image.cat([
    ee.Image(f"{root_asset}mapa_{m.lower()}_{reg_l}_{ver}").rename(f"pred_{m}")
    for m in modelos_evaluar
])

# Extraer predicciones de los 4 modelos exactamente en las coordenadas EPSG:4326 de CEO
gdf_ceo = gpd.GeoDataFrame(
    df_ceo[["point_uid", "stratum_id", "ref_id", COL_LON, COL_LAT]],
    geometry=[Point(xy) for xy in zip(df_ceo[COL_LON], df_ceo[COL_LAT])],
    crs="EPSG:4326",
)
fc_ceo = geemap.gdf_to_ee(gdf_ceo)
sampled = img_modelos.sampleRegions(
    collection=fc_ceo,
    properties=["point_uid", "stratum_id", "ref_id"],
    scale=10,
    tileScale=4,
    geometries=False,
)
df_val = pd.DataFrame([f["properties"] for f in sampled.getInfo()["features"]])
df_val = df_val.dropna(subset=[f"pred_{m}" for m in modelos_evaluar]).reset_index(drop=True)
for m in modelos_evaluar:
    df_val[f"pred_{m}"] = df_val[f"pred_{m}"].astype(int)

print(f"Puntos CEO totales: {len(df_ceo)} | Puntos validos cruzados con los 4 mapas: {len(df_val)}")
display(df_val.groupby("stratum_id").size().rename("n_muestras_estrato").to_frame())
df_val.to_csv(dirs["tablas"] / f"ceo_cruzado_4modelos_{reg_l}.csv", index=False)

In [ ]:
# 3. Calculo de precisiones insesgadas (Olofsson 2014 / Stehman 2014), F1-Score ± IC 95% y Area Ajustada (ha)
resultados_olof = {}
resumen_oe3 = []
conf = float(val_cfg["nivel_confianza"])
n_boot = int(val_cfg["bootstrap_iteraciones"])
seed = int(cfg["proyecto"]["seed"])

for m_name in modelos_evaluar:
    res = stehman_generalized_accuracy(
        df=df_val,
        stratum_col="stratum_id",
        pred_col=f"pred_{m_name}",
        ref_col="ref_id",
        stratum_areas_ha=STRATUM_AREAS_HA,
        class_labels=CLASS_IDS,
        class_names=CLASES_NOMBRES,
        confidence=conf,
        n_bootstrap=n_boot,
        seed=seed,
    )
    resultados_olof[m_name] = res
    m_row = res["per_class"].set_index("clase_id").loc[1]

    resumen_oe3.append({
        "Modelo": m_name,
        "N_CEO": res["n_muestras"],
        "OA": res["OA"],
        "OA_CI_half": res["OA_CI_half"],
        "Kappa": res["Kappa"],
        "Kappa_CI_half": 0.5 * (res["Kappa_CI_high"] - res["Kappa_CI_low"]),
        "PU_Mangle": m_row["PU"],
        "PU_Mangle_CI_half": m_row["PU_CI_half"],
        "PP_Mangle": m_row["PP"],
        "PP_Mangle_CI_half": m_row["PP_CI_half"],
        "F1_Mangle": m_row["F1"],
        "F1_Mangle_CI_half": 0.5 * (m_row["F1_CI_high"] - m_row["F1_CI_low"]),
        "Area_Mapeada_Mangle_ha": areas_json["areas_modelos_ha"][m_name].get("1", np.nan),
        "Area_Ajustada_Mangle_ha": m_row["area_ajustada_ha"],
        "Area_Ajustada_CI_half_ha": m_row["area_CI_half_ha"],
    })

df_oe3 = pd.DataFrame(resumen_oe3)
print(f"=== RESULTADOS OFICIALES OLOFSSON/STEHMAN (IC {int(conf*100)}%) — {REGION} ===")
display(df_oe3)

In [ ]:
# 4. Prueba formal de Hipotesis (H0 vs H1): McNemar, Prueba Z ponderada por diseno, Z de Kappa y ΔF1 Bootstrap
alpha = float(val_cfg["alpha_hipotesis"])
pares = [
    ("MS_SAR_PCA", "MS"),
    ("MS_SAR_PCA", "SAR"),
    ("MS_SAR_FULL", "MS"),
    ("MS_SAR_FULL", "SAR"),
    ("MS_SAR_PCA", "MS_SAR_FULL"),
]

filas_hip = []
for mod_a, mod_b in pares:
    mc_all = mcnemar_test(df_val["ref_id"], df_val[f"pred_{mod_a}"], df_val[f"pred_{mod_b}"], alpha=alpha)
    mc_mgl = mcnemar_test(df_val["ref_id"], df_val[f"pred_{mod_a}"], df_val[f"pred_{mod_b}"], binary_target_class=1, alpha=alpha)
    z_all = stratified_paired_z_test(df_val, "stratum_id", f"pred_{mod_a}", f"pred_{mod_b}", "ref_id", STRATUM_AREAS_HA, alpha=alpha)
    z_mgl = stratified_paired_z_test(df_val, "stratum_id", f"pred_{mod_a}", f"pred_{mod_b}", "ref_id", STRATUM_AREAS_HA, binary_target_class=1, alpha=alpha)
    kz = kappa_z_test(
        resultados_olof[mod_a]["Kappa"], resultados_olof[mod_a]["Kappa_SE"],
        resultados_olof[mod_b]["Kappa"], resultados_olof[mod_b]["Kappa_SE"],
        alpha=alpha,
    )
    # Diferencia pareada bootstrap de F1 Manglar (misma semilla estratificada -> muestras pareadas)
    diff_f1 = resultados_olof[mod_a]["bootstrap_f1"][:, 0] - resultados_olof[mod_b]["bootstrap_f1"][:, 0]
    f1_ci_lo, f1_ci_hi = np.nanpercentile(diff_f1, [2.5, 97.5])

    filas_hip.append({
        "Comparacion": f"{mod_a} vs {mod_b}",
        "Delta_OA_pct": z_all["delta_OA_A_menos_B"] * 100.0,
        "Z_diseno_OA": z_all["Z_stat"],
        "p_Z_OA": z_all["p_value"],
        "p_McNemar_multiclase": mc_all["p_value_recomendado"],
        "p_McNemar_Mangle": mc_mgl["p_value_recomendado"],
        "p_Z_diseno_Mangle": z_mgl["p_value"],
        "Delta_Kappa": kz["delta_kappa"],
        "p_Z_Kappa": kz["p_value"],
        "Delta_F1_Mangle_IC95": f"[{f1_ci_lo*100:+.2f}%, {f1_ci_hi*100:+.2f}%]",
        "Decision_H0_alpha005": ("Rechaza H0 (A > B significativo)" if z_all["delta_OA_A_menos_B"] > 0 else "Rechaza H0 (A < B significativo)") if (z_all["rechaza_H0"] or mc_mgl["rechaza_H0"]) else "No rechaza H0",
    })

df_hipotesis = pd.DataFrame(filas_hip)
print("=== CONTRASTE FORMAL DE HIPOTESIS (H0 vs H1) ===")
display(df_hipotesis)

In [ ]:
# 5. Figuras finales de tesis (300 DPI) y exportacion de Excel maestro del Objetivo 3
reg_nom = cfg["regiones"][REGION]["nombre_largo"]

fig_bar = plot_model_comparison(
    df_oe3,
    f"Comparación de Modelos (Validación Independiente Olofsson / CEO, IC 95%) — {reg_nom}",
    save_path=dirs["figuras"] / f"OE3_fig1_comparacion_modelos_olofsson_{reg_l}.png",
)
plt.show()

for m_name in modelos_evaluar:
    fig_p = plot_confusion_matrix(
        resultados_olof[m_name]["matrix_proportions"],
        f"Matriz de Confusión en Proporciones de Área (Olofsson) — {m_name} ({reg_nom})",
        is_proportion=True,
        save_path=dirs["figuras"] / f"OE3_cm_proporciones_{m_name.lower()}_{reg_l}.png",
    )
    plt.show()

# Grafico de comparacion de Area Mapeada vs Area Ajustada Olofsson (± IC 95%) vs MARN 2013
fig_area, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(df_oe3))
w = 0.35
ax.bar(x - w/2, df_oe3["Area_Mapeada_Mangle_ha"], width=w, label="Área mapeada (píxeles)", color="#78c679", edgecolor="#222")
ax.bar(
    x + w/2, df_oe3["Area_Ajustada_Mangle_ha"], width=w,
    yerr=df_oe3["Area_Ajustada_CI_half_ha"], capsize=5,
    label="Área ajustada insesgada (Olofsson ± IC 95%)", color="#006837", edgecolor="#222",
)
marn_base = float(cfg["regiones"][REGION]["linea_base_marn_2013_ha"])
ax.axhline(marn_base, color="#d7191c", linestyle="--", linewidth=1.4, label=f"Referencia MARN (2013): {marn_base:,.1f} ha")
ax.set_xticks(x)
ax.set_xticklabels(df_oe3["Modelo"], fontweight="bold")
ax.set_ylabel("Superficie de bosque manglar (ha)")
ax.set_title(f"Estimación de Superficie de Manglar e Incertidumbre (IC 95%) — {reg_nom}", fontweight="bold")
ax.legend(frameon=True)
fig_area.tight_layout()
fig_area.savefig(dirs["figuras"] / f"OE3_fig2_area_manglar_olofsson_{reg_l}.png", dpi=300)
plt.show()

# Exportar libro Excel final del Objetivo 3 e Hipotesis
xlsx_oe3 = dirs["tablas"] / f"OE3_Validacion_Olofsson_Hipotesis_{reg_l}.xlsx"
with pd.ExcelWriter(xlsx_oe3, engine="openpyxl") as writer:
    df_oe3.to_excel(writer, sheet_name="Resumen_Modelos_OE3", index=False)
    df_hipotesis.to_excel(writer, sheet_name="Prueba_Hipotesis", index=False)
    for m_name in modelos_evaluar:
        resultados_olof[m_name]["per_class"].to_excel(writer, sheet_name=f"Metricas_{m_name}", index=False)
        resultados_olof[m_name]["matrix_proportions"].to_excel(writer, sheet_name=f"CM_Prop_{m_name}")
        resultados_olof[m_name]["matrix_counts"].to_excel(writer, sheet_name=f"CM_Cont_{m_name}")

print(f"[OK] Entregables finales exportados en: {xlsx_oe3}")
print(f"[OK] Figuras 300 DPI guardadas en     : {dirs['figuras']}")